Standardize column names and correct data types by implementing a dynamic parser that automatically detects and converts incorrect formats, ensuring consistency across the entire dataset without introducing errors.

In [40]:
import pandas as pd
import numpy as np
import re

In [41]:
df = pd.read_excel('Womens Clothing Reviews Data.xlsx')
df

,Product ID,Category,Subcategory1,SubCategory2,Location,Channel,Customer Age,Review Title,Review Text,Rating,Recommend Flag
0,767,Initmates,Intimate,Intimates,Mumbai,Mobile,33,NaN,Absolutely wonderful - silky and sexy and comfortable,4,1
1,1080,General,Dresses,Dresses,Bangalore,Mobile,34,NaN,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,1
2,1077,General,Dresses,Dresses,Gurgaon,Mobile,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c",3,0
3,1049,General Petite,Bottoms,Pants,Chennai,Web,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",5,1
4,847,General,Tops,Blouses,Bangalore,Web,47,Flattering shirt,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,5,1
...,...,...,...,...,...,...,...,...,...,...,...
23481,1104,General Petite,Dresses,Dresses,Gurgaon,Mobile,34,Great dress for many occasions,I was very happy to snag this dress at such a great price! it's very easy to slip on and has a very flattering cut and color combo.,5,1
23482,862,General Petite,Tops,Knits,Gurgaon,Web,48,Wish it was made of cotton,"It reminds me of maternity clothes. soft, stretchy, shiny material. cut is flattering and drapes nicely. i only found one button to close front... looked awkward. nice long sleeves.\nnot for me but maybe for others. just ok.",3,1
23483,1104,General Petite,Dresses,Dresses,Gurgaon,Web,31,"Cute, but see through","This fit well, but the top was very see through. this never would have worked for me. i'm glad i was able to try it on in the store and didn't order it online. with different fabric, it would have been great.",3,0
23484,1084,General,Dresses,Dresses,Gurgaon,Web,28,"Very cute dress, perfect for summer parties and we","I bought this dress for a wedding i have this summer, and it's so cute. unfortunately the fit isn't perfect. the medium fits my waist perfectly, but was way too long and too big in the bust and shoulders. if i wanted to spend the money, i could get it tailored, but i just felt like it might not be worth it. side note - this dress was delivered to me with a nordstrom tag on it and i found it much cheaper there after looking!",3,1


In [42]:
df.shape

(23486, 11)

In [43]:
print(df.columns.tolist())

['Product ID', 'Category', 'Subcategory1', 'SubCategory2', 'Location', 'Channel', 'Customer Age', 'Review Title', 'Review Text', 'Rating', 'Recommend Flag']


Standerdize Column Names

In [44]:
df.columns = (df.columns.str.strip().str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
                
print(df.columns.tolist())

['product_id', 'category', 'subcategory1', 'subcategory2', 'location', 'channel', 'customer_age', 'review_title', 'review_text', 'rating', 'recommend_flag']


In [45]:
df.columns.is_unique

True

In [46]:
df.dtypes

product_id         int64
category          object
subcategory1      object
subcategory2      object
location          object
channel           object
customer_age       int64
review_title      object
review_text       object
rating             int64
recommend_flag     int64
dtype: object

In [47]:
df.isnull().sum()

product_id           0
category            14
subcategory1        14
subcategory2        14
location             0
channel              0
customer_age         0
review_title      3810
review_text        845
rating               0
recommend_flag       0
dtype: int64

In [48]:
df.isnull().sum() / len(df) * 100

product_id         0.000000
category           0.059610
subcategory1       0.059610
subcategory2       0.059610
location           0.000000
channel            0.000000
customer_age       0.000000
review_title      16.222430
review_text        3.597888
rating             0.000000
recommend_flag     0.000000
dtype: float64

Dynamically detect numeric columns stored as text

In [49]:

numeric_pattern = r'(^|_)(id|age|rating|flag|count|price|quantity|number)(_|$)'

for col in df.columns:

    # Skip text columns
    if re.search(r'review|title|text|comment|description', col):
        continue

    # Process only columns whose names indicate numeric data
    if re.search(numeric_pattern, col):

        original = df[col].copy()

        cleaned = (
            original.astype('string')
            .str.strip()
            .str.replace(',', '', regex=False)
        )

        converted = pd.to_numeric(cleaned, errors='coerce')

        invalid = original.notna() & converted.isna()

        print(f"Column: {col}, Invalid entries: {invalid.sum()}")

        # Convert only when all existing values are valid
        if invalid.sum() == 0:

            if (converted.dropna() % 1 == 0).all():
                df[col] = converted.astype('Int64')
            else:
                df[col] = converted.astype('Float64')

        else:
            print(f"Skipped conversion for {col} due to invalid values.")

print("\nUpdated data types:")
print(df.dtypes)

Column: product_id, Invalid entries: 0
Column: customer_age, Invalid entries: 0
Column: rating, Invalid entries: 0
Column: recommend_flag, Invalid entries: 0

Updated data types:
product_id         Int64
category          object
subcategory1      object
subcategory2      object
location          object
channel           object
customer_age       Int64
review_title      object
review_text       object
rating             Int64
recommend_flag     Int64
dtype: object


In [50]:
for col in df.columns:
    print(f"\nColumn: {col}")
    print(df[col].head(10).tolist())


Column: product_id
[767, 1080, 1077, 1049, 847, 1080, 858, 858, 1077, 1077]

Column: category
['Initmates', 'General', 'General', 'General Petite', 'General', 'General', 'General Petite', 'General Petite', 'General', 'General']

Column: subcategory1
['Intimate', 'Dresses', 'Dresses', 'Bottoms', 'Tops', 'Dresses', 'Tops', 'Tops', 'Dresses', 'Dresses']

Column: subcategory2
['Intimates', 'Dresses', 'Dresses', 'Pants', 'Blouses', 'Dresses', 'Knits', 'Knits', 'Dresses', 'Dresses']

Column: location
['Mumbai', 'Bangalore', 'Gurgaon', 'Chennai', 'Bangalore', 'Gurgaon', 'Gurgaon', 'Gurgaon', 'Mumbai', 'Gurgaon']

Column: channel
['Mobile', 'Mobile', 'Mobile', 'Web', 'Web', 'Web', 'Mobile', 'Mobile', 'Web', 'Mobile']

Column: customer_age
[33, 34, 60, 50, 47, 49, 39, 39, 24, 34]

Column: review_title
[nan, nan, 'Some major design flaws', 'My favorite buy!', 'Flattering shirt', 'Not for the very petite', 'Cagrcoal shimmer fun', 'Shimmer, surprisingly goes with lots', 'Flattering', 'Such a fun 

In [38]:
df.dtypes

product_id           Int64
category          category
subcategory1      category
subcategory2      category
location          category
channel           category
customer_age         Int64
review_title        object
review_text         object
rating               Int64
recommend_flag       Int64
dtype: object

In [51]:
df['category'].value_counts()

category
General           13850
General Petite     8120
Initmates          1502
Name: count, dtype: int64

In [52]:
df.dtypes

product_id         Int64
category          object
subcategory1      object
subcategory2      object
location          object
channel           object
customer_age       Int64
review_title      object
review_text       object
rating             Int64
recommend_flag     Int64
dtype: object

Standardize categorical columns

In [58]:
categorical_columns = ['category', 'subcategory1', 'subcategory2', 'location', 'channel']
print(categorical_columns)

for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].astype('category')

print('\n\n', df.dtypes)

['category', 'subcategory1', 'subcategory2', 'location', 'channel']


 product_id           Int64
category          category
subcategory1      category
subcategory2      category
location          category
channel           category
customer_age         Int64
review_title        object
review_text         object
rating               Int64
recommend_flag       Int64
dtype: object
